<a href="https://colab.research.google.com/github/fedeiria/tpi-analisis-de-datos/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Práctico Integrador — Introducción al Análisis de Datos
## Predicción de cancelación de clientes (Churn) en una empresa de telecomunicaciones

**UTN — Facultad Regional Avellaneda** · 2º Cuatrimestre 2026 · Docente: Enzo Martín Zotti

| Integrante | División |
|---|---|
| Nombre Apellido | — |
| Nombre Apellido | — |
| Nombre Apellido | — |
| Nombre Apellido | — |

## Índice

0. Configuración del entorno
1. **A — Fundamentos** (problema de negocio, hipótesis, clasificación de variables)
2. **A — Manipulación de datos** (selección, filtrado, agregación, transformación)
3. **A — Limpieza y calidad de datos** (nulos, dtypes, outliers)
4. **B — Análisis exploratorio y estadística aplicada**
5. **B — Visualización de datos**
6. **C — Preprocesamiento para modelado**
7. **C — Regresión lineal simple**
8. **C — Clasificación (Churn)**
9. Conclusiones

> **Convención del equipo:** cada sección indica su responsable. Cada gráfico, tabla o resultado va seguido de una celda markdown con la interpretación orientada al negocio.

---
# 0. Configuración del entorno

In [ ]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (r2_score, mean_absolute_error,
                             confusion_matrix, ConfusionMatrixDisplay, accuracy_score)

# Configuración general
RANDOM_STATE = 42          # semilla única para todo el notebook (reproducibilidad)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

### Carga del dataset

Fuente original: [Telco Customer Churn — Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).
El CSV se versiona dentro del repositorio (`data/raw/`) y se lee desde la URL *raw* de GitHub, para que el notebook corra en Colab sin credenciales de Kaggle.

In [ ]:
# URL del CSV dentro de nuestro repositorio (completar cuando el repo exista)
DATA_URL = "https://raw.githubusercontent.com/fedeiria/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Copia pública de respaldo del mismo dataset (repositorio de IBM)
FALLBACK_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    df_raw = pd.read_csv(DATA_URL)
    print("Dataset cargado desde el repositorio del grupo.")
except Exception:
    df_raw = pd.read_csv(FALLBACK_URL)
    print("Dataset cargado desde la URL de respaldo.")

df = df_raw.copy()   # trabajamos sobre una copia; df_raw queda intacto como referencia
print(df.shape)
df.head()

---
# 1. A — Fundamentos
**Responsable:** Federico Iriarte

### 1.1 Problema de negocio

Una empresa de telecomunicaciones ofrece telefonía, internet y servicios adicionales (seguridad, soporte técnico, streaming) bajo distintos tipos de contrato. Una parte de sus clientes **cancela el servicio** y se va a la competencia: a este fenómeno se lo conoce como **churn** (abandono o fuga de clientes).

El churn impacta directamente en los ingresos recurrentes de la empresa y, además, conseguir un cliente nuevo suele ser bastante más caro que retener a uno existente (publicidad, promociones de bienvenida, costos de instalación). Por eso, a la empresa le conviene **detectar a tiempo qué clientes tienen mayor riesgo de irse**, para dirigirles acciones de retención (descuentos, migración a contratos más largos, mejoras en el servicio) antes de que cancelen.

**Pregunta de negocio:** ¿qué características de los clientes y de sus contratos están asociadas a la cancelación, y es posible anticipar qué clientes van a cancelar a partir de esos datos?

### 1.2 Variable objetivo

La variable objetivo es **`Churn`**, que indica si el cliente canceló el servicio en el último mes. Es una variable **cualitativa nominal dicotómica** (dos categorías: `Yes` / `No`). Para el modelado se codificará como binaria (`1` = canceló, `0` = permanece).

In [ ]:
# Distribución de la variable objetivo: frecuencia absoluta y relativa
churn_dist = pd.DataFrame({
    "clientes": df["Churn"].value_counts(),
    "porcentaje": (df["Churn"].value_counts(normalize=True) * 100).round(2),
})
churn_dist

**Interpretación:** aproximadamente **1 de cada 4 clientes (26,5%) canceló** el servicio. Las clases están **desbalanceadas** (73,5% vs. 26,5%): esto se tendrá en cuenta al particionar los datos (partición estratificada) y al interpretar la accuracy del modelo de clasificación, ya que un modelo que predijera siempre "No" acertaría el 73,5% de las veces sin aprender nada útil.

### 1.3 Hipótesis analíticas

A partir del conocimiento del negocio se plantean las siguientes hipótesis, que se contrastarán con los datos en la sección 4:

- **H1 — Tipo de contrato:** los clientes con contrato **mes a mes** cancelan en mayor proporción que los de contratos de uno o dos años, porque no tienen un compromiso de permanencia que los retenga.
- **H2 — Antigüedad:** los clientes con **menor antigüedad** (`tenure`) cancelan más, porque todavía no consolidaron una relación con la empresa.
- **H3 — Método de pago:** los clientes que pagan con **cheque electrónico** cancelan más que los que tienen pagos automáticos (transferencia o tarjeta), porque el pago manual implica una decisión activa cada mes.
- **H4 — Cargo mensual:** los clientes con **cargos mensuales más altos** cancelan más, porque perciben un mayor costo y tienen más incentivo para buscar alternativas.

### 1.4 Clasificación del dataset

El dataset es **estructurado**: los datos están organizados en una tabla con filas y columnas de esquema fijo, donde cada **fila representa un cliente** (registro) y cada **columna un atributo** con un significado y tipo definidos. Se almacena en formato CSV y puede manipularse directamente con Pandas o SQL.

No es *semi-estructurado* (como JSON o XML, donde los registros pueden tener distintos campos o anidamientos) ni *no estructurado* (como texto libre, imágenes o audio, que no tienen un esquema tabular predefinido).

In [ ]:
# Vista general de la estructura: filas, columnas, dtypes y nulos detectados por Pandas
df.info()

**Interpretación:** el dataset tiene **7.043 registros y 21 columnas**, y Pandas no detecta valores nulos explícitos. Sin embargo, la columna `TotalCharges` (un importe en dinero) fue leída como **texto** en lugar de número, lo que anticipa un problema de calidad que se analiza en la sección 3.

### 1.5 Clasificación de variables: teoría vs. dtypes de Pandas

Se clasifica cada variable según la teoría estadística y se contrasta con el tipo de dato (*dtype*) que Pandas asignó automáticamente al leer el CSV. La columna `coincide` indica si el dtype es coherente con el tipo teórico (numérico para las cuantitativas, texto o categoría para las cualitativas).

In [ ]:
# Clasificación teórica de cada variable
tipo_teorico = {
    "customerID":       "Identificador",
    "gender":           "Cualitativa nominal",
    "SeniorCitizen":    "Cualitativa nominal",
    "Partner":          "Cualitativa nominal",
    "Dependents":       "Cualitativa nominal",
    "tenure":           "Cuantitativa discreta",
    "PhoneService":     "Cualitativa nominal",
    "MultipleLines":    "Cualitativa nominal",
    "InternetService":  "Cualitativa nominal",
    "OnlineSecurity":   "Cualitativa nominal",
    "OnlineBackup":     "Cualitativa nominal",
    "DeviceProtection": "Cualitativa nominal",
    "TechSupport":      "Cualitativa nominal",
    "StreamingTV":      "Cualitativa nominal",
    "StreamingMovies":  "Cualitativa nominal",
    "Contract":         "Cualitativa ordinal",
    "PaperlessBilling": "Cualitativa nominal",
    "PaymentMethod":    "Cualitativa nominal",
    "MonthlyCharges":   "Cuantitativa continua",
    "TotalCharges":     "Cuantitativa continua",
    "Churn":            "Cualitativa nominal (objetivo)",
}

clasificacion = pd.DataFrame({
    "variable": df.columns,
    "tipo_teorico": [tipo_teorico[c] for c in df.columns],
    "dtype_pandas": df.dtypes.astype(str).values,
    "valores_unicos": df.nunique().values,
})

# ¿El dtype es coherente con el tipo teórico?
def es_coherente(fila):
    es_numerico = pd.api.types.is_numeric_dtype(df[fila["variable"]])
    if fila["tipo_teorico"].startswith("Cuantitativa"):
        return es_numerico
    return not es_numerico   # cualitativas e identificador deberían ser texto/categoría

clasificacion["coincide"] = clasificacion.apply(es_coherente, axis=1)
clasificacion

In [ ]:
# Variables donde la teoría y el dtype de Pandas NO coinciden
clasificacion[~clasificacion["coincide"]]

**Interpretación:**

La mayoría de las variables son **cualitativas nominales** (servicios contratados y datos demográficos con categorías como `Yes`/`No`), y Pandas las leyó como texto (`object`; en versiones recientes de Pandas aparece como `str`), lo cual es coherente.

- **`Contract`** se clasifica como **ordinal** porque sus categorías tienen un orden natural según la duración del compromiso: *Month-to-month* < *One year* < *Two year*.
- **`tenure`** (meses de antigüedad) es **cuantitativa discreta**: toma valores enteros contables. **`MonthlyCharges`** y **`TotalCharges`** son **cuantitativas continuas** (importes de dinero con decimales).
- **`customerID`** es un **identificador**: distingue a cada cliente pero no aporta información analítica, por lo que se excluirá del modelado.

Se detectan **dos discrepancias** entre la teoría y los dtypes:

1. **`SeniorCitizen`** es conceptualmente cualitativa (el cliente es o no es adulto mayor), pero Pandas la leyó como `int64` porque viene **codificada como 0/1**. Calcular su promedio o desvío no tiene sentido como medida de una cantidad, por lo que se convertirá a categoría.
2. **`TotalCharges`** es cuantitativa continua, pero Pandas la leyó como **texto**. Esto indica que la columna contiene algún valor no numérico que impidió la conversión automática. Se investiga y corrige en la sección 3.

---
# 2. A — Manipulación de datos
**Responsable:** Federico Iriarte

### 2.1 Selección y filtrado con múltiples condiciones

Se definen filtros que responden preguntas de negocio concretas y se compara la tasa de cancelación de cada grupo contra la tasa general (26,5%). Para combinar condiciones se usan los operadores `&` (Y lógico) y `|` (O lógico), con cada condición entre paréntesis.

In [ ]:
def tasa_churn(datos: pd.DataFrame) -> float:
    """Porcentaje de clientes que cancelaron dentro de un subconjunto."""
    return round((datos["Churn"] == "Yes").mean() * 100, 2)

print(f"Tasa de churn general: {tasa_churn(df)}%")

In [ ]:
# Filtro 1 (&): clientes con contrato mes a mes Y menos de 12 meses de antigüedad
nuevos_sin_compromiso = df[(df["Contract"] == "Month-to-month") & (df["tenure"] < 12)]

print(f"Clientes: {len(nuevos_sin_compromiso)} ({len(nuevos_sin_compromiso) / len(df):.1%} del total)")
print(f"Tasa de churn: {tasa_churn(nuevos_sin_compromiso)}%")

# Selección de columnas relevantes con .loc para inspeccionar el grupo
nuevos_sin_compromiso.loc[:, ["customerID", "tenure", "Contract", "MonthlyCharges", "Churn"]].head()

In [ ]:
# Filtro 2 (|): clientes con pago manual (cheque electrónico O cheque por correo)
pago_manual = df[(df["PaymentMethod"] == "Electronic check") | (df["PaymentMethod"] == "Mailed check")]
pago_automatico = df[~df.index.isin(pago_manual.index)]   # el complemento: pagos automáticos

print(f"Pago manual     -> clientes: {len(pago_manual)}, tasa de churn: {tasa_churn(pago_manual)}%")
print(f"Pago automático -> clientes: {len(pago_automatico)}, tasa de churn: {tasa_churn(pago_automatico)}%")

In [ ]:
# Filtro 3 (& y | combinados): clientes con fibra óptica, sin soporte técnico
# y que además tienen contrato mes a mes O cargo mensual mayor a 90
riesgo_fibra = df[
    (df["InternetService"] == "Fiber optic")
    & (df["TechSupport"] == "No")
    & ((df["Contract"] == "Month-to-month") | (df["MonthlyCharges"] > 90))
]
print(f"Clientes: {len(riesgo_fibra)}, tasa de churn: {tasa_churn(riesgo_fibra)}%")

**Interpretación:**

- Los clientes **nuevos con contrato mes a mes** representan el 27% de la cartera y cancelan en un **51,9%**, prácticamente el doble de la tasa general (26,5%). Es el segmento de mayor riesgo y un primer indicio a favor de **H1** y **H2**.
- Los clientes con **pago manual** cancelan un **34,7%**, más del doble que los que tienen pago automático (**16,0%**), lo que va en línea con **H3**.
- Combinando condiciones se aísla un segmento de 2.141 clientes (fibra óptica, sin soporte técnico y con contrato flexible o cargo alto) con una tasa de churn del **51,0%**. Este tipo de filtro permite definir **grupos objetivo concretos para campañas de retención**.

### 2.2 Agregación con `groupby`

Se segmenta la cartera de clientes combinando varias agregaciones en una sola operación (*named aggregation*): cantidad de clientes, promedios y tasa de cancelación por grupo.

In [ ]:
# Segmentación por tipo de contrato
resumen_contrato = df.groupby("Contract").agg(
    clientes=("customerID", "count"),
    antiguedad_prom=("tenure", "mean"),
    cargo_mensual_prom=("MonthlyCharges", "mean"),
    cargo_mensual_total=("MonthlyCharges", "sum"),
    tasa_churn=("Churn", lambda s: (s == "Yes").mean() * 100),
).round(2)
resumen_contrato

In [ ]:
# Segmentación por dos variables: contrato y tipo de servicio de internet
resumen_contrato_internet = df.groupby(["Contract", "InternetService"]).agg(
    clientes=("customerID", "count"),
    cargo_mensual_prom=("MonthlyCharges", "mean"),
    tasa_churn=("Churn", lambda s: (s == "Yes").mean() * 100),
).round(2)
resumen_contrato_internet

**Interpretación:**

- La tasa de churn **cae fuertemente a medida que el contrato es más largo**: **42,7%** en el contrato mes a mes, **11,3%** en el de un año y apenas **2,8%** en el de dos años. Los clientes con contratos largos también tienen, en promedio, mucha más antigüedad.
- La columna `cargo_mensual_total` muestra cuánto **ingreso mensual** representa cada segmento: el contrato mes a mes es a la vez el segmento más numeroso y el más inestable, por lo que concentra el mayor ingreso en riesgo.
- Al cruzar con el tipo de internet se ve que, dentro de cada tipo de contrato, los clientes con **fibra óptica** cancelan más que los de DSL (por ejemplo, en el contrato mes a mes: **54,6%** contra **32,2%**), a pesar de pagar más. Esto sugiere un posible problema de satisfacción con ese servicio (precio o calidad), que la empresa debería investigar.

### 2.3 Transformación básica

Se aplican tres transformaciones:

1. **Eliminación de atributos irrelevantes:** se elimina `customerID`, ya que es un identificador único que no aporta información sobre el comportamiento del cliente. Si se usara como predictor, el modelo podría "memorizar" clientes en lugar de aprender patrones generalizables.
2. **Renombrado de columnas:** el dataset mezcla convenciones de nombres (`gender` y `tenure` en minúscula, el resto en *PascalCase*). Se renombran esas dos columnas para que todas sigan el mismo estándar.
3. **Columnas calculadas:** se crean nuevas variables a partir de las existentes.

In [ ]:
# 1. Eliminación del identificador
df = df.drop(columns=["customerID"])

# 2. Renombrado para unificar la convención de nombres (PascalCase)
df = df.rename(columns={"gender": "Gender", "tenure": "Tenure"})

df.columns.tolist()

In [ ]:
# 3. Columnas calculadas

# Cantidad de servicios contratados por cliente
servicios_adicionales = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                         "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df["CantidadServicios"] = (
    (df[servicios_adicionales] == "Yes").sum(axis=1)       # servicios con valor "Yes"
    + (df["InternetService"] != "No").astype(int)           # +1 si tiene internet (DSL o fibra)
)

# Cargo mensual por servicio: operación aritmética entre dos variables
df["CargoPorServicio"] = (df["MonthlyCharges"] / df["CantidadServicios"]).round(2)

# Discretización de la antigüedad en rangos (útil para el EDA y la hipótesis H2)
df["GrupoAntiguedad"] = pd.cut(
    df["Tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12 meses", "13-24 meses", "25-48 meses", "49-72 meses"],
)

df[["Tenure", "GrupoAntiguedad", "MonthlyCharges", "CantidadServicios", "CargoPorServicio"]].head()

**Interpretación:** `CantidadServicios` resume en un solo número el grado de "involucramiento" del cliente con la empresa (va de 1 a 9 servicios), y `CargoPorServicio` permite comparar cuánto paga cada cliente por cada servicio que usa. `GrupoAntiguedad` transforma una variable cuantitativa en una **cualitativa ordinal**, lo que facilita comparar tasas de cancelación por rangos de antigüedad.

### 2.4 Integración de datos

Se integra al dataset una **tabla de referencia externa** con la duración de cada tipo de contrato en meses, mediante un `merge` (equivalente a un `LEFT JOIN` en SQL) usando `Contract` como clave. Esta información no está en el dataset original, pero es conocimiento del negocio que convierte el contrato en una magnitud comparable.

In [ ]:
# Tabla de referencia: duración del compromiso según el tipo de contrato
tabla_contratos = pd.DataFrame({
    "Contract": ["Month-to-month", "One year", "Two year"],
    "DuracionContratoMeses": [1, 12, 24],
})

filas_antes = len(df)
df = df.merge(tabla_contratos, on="Contract", how="left", validate="many_to_one")

# Control de calidad de la integración: no se deben perder ni duplicar filas,
# y todos los clientes deben haber encontrado su contrato en la tabla
print(f"Filas antes: {filas_antes} | Filas después: {len(df)}")
print(f"Clientes sin coincidencia: {df['DuracionContratoMeses'].isna().sum()}")

df[["Contract", "DuracionContratoMeses"]].drop_duplicates().sort_values("DuracionContratoMeses")

**Interpretación:** la integración se realizó sin pérdida ni duplicación de registros (`validate="many_to_one"` garantiza que cada contrato aparece una sola vez en la tabla de referencia) y todos los clientes encontraron coincidencia. La nueva columna expresa numéricamente el **orden natural** de la variable ordinal `Contract`.

---
# 3. A — Limpieza y calidad de datos
**Responsable:** Federico Iriarte

### 3.1 Valores faltantes, cadenas anómalas y duplicados

In [ ]:
# Nulos explícitos (NaN) por columna
print(f"Nulos explícitos en todo el dataset: {df.isna().sum().sum()}")

# Cadenas vacías o compuestas solo por espacios en las columnas de texto
columnas_texto = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])
                  and c != "GrupoAntiguedad"]
vacios = {c: (df[c].astype(str).str.strip() == "").sum() for c in columnas_texto}
pd.Series(vacios, name="cadenas_vacias")[lambda s: s > 0]

In [ ]:
# ¿Quiénes son los clientes con TotalCharges vacío?
mascara_vacios = df["TotalCharges"].astype(str).str.strip() == ""
df.loc[mascara_vacios, ["Tenure", "Contract", "MonthlyCharges", "TotalCharges", "Churn"]]

**Interpretación y decisión:** Pandas no detectó nulos porque los faltantes de `TotalCharges` no son `NaN`, sino **cadenas con un espacio en blanco** (`" "`). Son 11 registros (0,16% del total) y **todos tienen `Tenure = 0`**: son clientes que recién se dieron de alta y todavía no recibieron su primera factura.

Por lo tanto, el faltante no es un error aleatorio sino que tiene una explicación de negocio: el importe acumulado de estos clientes es **cero**. Se decide **imputar `TotalCharges = 0`** en lugar de eliminar los registros, porque refleja la realidad y conserva clientes válidos. (Eliminarlos tampoco habría sido grave dado su bajo volumen, pero se perdería información sin necesidad.)

In [ ]:
# Registros duplicados
print(f"Filas duplicadas (sin customerID): {df.duplicated().sum()}")
print(f"customerID duplicados en el dataset original: {df_raw['customerID'].duplicated().sum()}")

**Interpretación:** aparecen filas idénticas una vez eliminado el `customerID`, pero en el dataset original **cada una tiene un identificador distinto**. No son registros cargados dos veces, sino **clientes diferentes con el mismo perfil** (por ejemplo, clientes con solo telefonía básica y la misma antigüedad). Por eso se conservan: eliminarlos subrepresentaría a los perfiles más comunes.

### 3.2 Adecuación de tipos de datos

Se corrigen los dtypes que no coinciden con la clasificación teórica de la sección 1.5:

- `TotalCharges`: texto → numérico (`float64`), aplicando la imputación decidida en 3.1.
- `SeniorCitizen`: entero `0/1` → categoría `No/Yes`, igual que el resto de las variables binarias.
- Variables cualitativas nominales: texto → `category`, el tipo de Pandas para variables con un conjunto acotado de valores (además reduce el uso de memoria).
- `Contract`: texto → `category` **ordenada**, porque es ordinal.

In [ ]:
dtypes_antes = df.dtypes.astype(str)
memoria_antes = df.memory_usage(deep=True).sum() / 1024**2

# TotalCharges: espacios -> NaN -> número; luego imputación con 0 (clientes con Tenure = 0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(r"^\s*$", np.nan, regex=True))
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# SeniorCitizen: 0/1 -> No/Yes como categoría
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"}).astype("category")

# Cualitativas nominales -> category
nominales = ["Gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
             "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
             "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling",
             "PaymentMethod", "Churn"]
df[nominales] = df[nominales].astype("category")

# Contract -> categoría ordenada (ordinal)
df["Contract"] = pd.Categorical(df["Contract"],
                                categories=["Month-to-month", "One year", "Two year"],
                                ordered=True)

memoria_despues = df.memory_usage(deep=True).sum() / 1024**2

comparacion_dtypes = pd.DataFrame({"antes": dtypes_antes, "después": df.dtypes.astype(str)})
print(f"Memoria: {memoria_antes:.2f} MB -> {memoria_despues:.2f} MB")
comparacion_dtypes[comparacion_dtypes["antes"] != comparacion_dtypes["después"]]

In [ ]:
# Verificación: ya no quedan nulos y TotalCharges es numérico
print(f"Nulos restantes: {df.isna().sum().sum()}")
print(f"TotalCharges de clientes con Tenure = 0: {df.loc[df['Tenure'] == 0, 'TotalCharges'].unique()}")

**Interpretación:** todas las variables quedaron con un dtype coherente con su clasificación teórica, no quedan valores faltantes y el uso de memoria se redujo considerablemente gracias al tipo `category`. A partir de aquí `TotalCharges` puede usarse en cálculos estadísticos.

### 3.3 Valores atípicos (outliers)

Se analizan las tres variables cuantitativas con **boxplots** y con el **método del rango intercuartílico (IQR)**: se considera atípico todo valor fuera del intervalo [Q1 − 1,5·IQR ; Q3 + 1,5·IQR].

In [ ]:
def outliers_iqr(serie: pd.Series) -> pd.Series:
    """Devuelve una máscara booleana con los valores fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)

numericas = ["Tenure", "MonthlyCharges", "TotalCharges"]

resumen_outliers = pd.DataFrame({
    "Q1": [df[c].quantile(0.25) for c in numericas],
    "Q3": [df[c].quantile(0.75) for c in numericas],
    "límite_inferior": [df[c].quantile(0.25) - 1.5 * (df[c].quantile(0.75) - df[c].quantile(0.25)) for c in numericas],
    "límite_superior": [df[c].quantile(0.75) + 1.5 * (df[c].quantile(0.75) - df[c].quantile(0.25)) for c in numericas],
    "outliers": [outliers_iqr(df[c]).sum() for c in numericas],
}, index=numericas).round(2)
resumen_outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numericas):
    sns.boxplot(y=df[col], ax=ax, color="steelblue")
    ax.set_title(f"Boxplot de {col}")
    ax.set_ylabel(col)
plt.tight_layout()
plt.show()

**Interpretación y decisión:** según el criterio IQR **no hay valores atípicos** en ninguna de las tres variables; los límites calculados quedan fuera del rango de los datos (por ejemplo, el límite inferior es negativo, lo que no es posible para antigüedades ni importes).

`TotalCharges` muestra una **asimetría positiva** (la caja está desplazada hacia abajo y el bigote superior es largo): hay muchos clientes con importes acumulados bajos y pocos con importes muy altos. Esto es esperable, porque `TotalCharges` es aproximadamente el producto entre la antigüedad y el cargo mensual, y los clientes antiguos con planes caros acumulan montos altos. Son valores **legítimos**, no errores, por lo que **se conservan todos los registros**.

### 3.4 Guardado del dataset depurado

El dataset limpio y transformado se guarda en `data/processed/`, según la estructura del repositorio.

In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/telco_churn_clean.csv", index=False)
print(f"Dataset depurado guardado: {df.shape[0]} filas x {df.shape[1]} columnas")

---
# 4. B — Análisis exploratorio y estadística aplicada
**Responsable:** _a definir_

### 4.1 Análisis cuantitativo
_Medidas de tendencia central y dispersión de `tenure`, `MonthlyCharges` y `TotalCharges`, con interpretación._

In [ ]:
# TODO: describe() + mediana, moda, rango, desvío, coeficiente de variación

### 4.2 Correlación lineal (Pearson / Spearman)
_Calcular ambas, comparar y explicar cuándo conviene cada una._

In [ ]:
# TODO: df[[...]].corr(method="pearson") y method="spearman"

### 4.3 Análisis categórico
_Frecuencias absolutas y relativas, y tablas de contingencia de variables sociodemográficas y de contratación contra `Churn`._

In [ ]:
# TODO: value_counts() y value_counts(normalize=True)
# TODO: pd.crosstab(..., df["Churn"], normalize="index")

### 4.4 Contraste de hipótesis planteadas
_Volver a las hipótesis H1–H4 de la sección 1.3 e indicar si los datos las respaldan._

---
# 5. B — Visualización de datos
**Responsable:** _a definir_

> Cada gráfico va seguido de una celda con su interpretación orientada al negocio.

### 5.1 Distribución de `tenure` y `MonthlyCharges` (histograma + KDE)

In [ ]:
# TODO

_Interpretación:_

### 5.2 Matriz de correlación (heatmap)

In [ ]:
# TODO

_Interpretación:_

### 5.3 Tasa de Churn por tipo de contrato

In [ ]:
# TODO

_Interpretación:_

### 5.4 Tasa de Churn por método de pago

In [ ]:
# TODO

_Interpretación:_

---
# 6. C — Preprocesamiento para modelado
**Responsable:** _a definir_

> **Regla anti data leakage:** primero se hace la partición train/test; los encoders y scalers se ajustan (`fit`) **solo con train** y después se aplican (`transform`) a test. Un `Pipeline` con `ColumnTransformer` garantiza esto automáticamente.

### 6.1 Definición de X e y

In [ ]:
# TODO: definir features (X) y objetivo (y), excluyendo identificadores

### 6.2 Partición estratificada 80/20

In [ ]:
# TODO: train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

### 6.3 Codificación y escalado
_Justificar qué variables van con One-Hot y cuáles con Ordinal, y por qué StandardScaler o MinMaxScaler._

In [ ]:
# TODO: ColumnTransformer con OneHotEncoder / OrdinalEncoder y StandardScaler / MinMaxScaler

---
# 7. C — Predicción numérica: regresión lineal simple
**Responsable:** _a definir_

_Estimar una variable continua (ej.: `TotalCharges`) a partir de **una** variable predictora. Justificar la elección del predictor._

In [ ]:
# TODO: entrenar LinearRegression con una sola feature
# TODO: evaluar con r2_score y mean_absolute_error sobre test

_Interpretación de coeficientes y métricas:_

---
# 8. C — Predicción categórica: clasificación de Churn
**Responsable:** _a definir_

_Entrenar **un** modelo elemental (Regresión Logística o Árbol de Decisión) y justificar la elección._

In [ ]:
# TODO: Pipeline(preprocesador + clasificador), fit sobre train
# TODO: predecir sobre test

In [ ]:
# TODO: matriz de confusión (ConfusionMatrixDisplay) y accuracy_score

_Interpretación: qué significan los falsos negativos y falsos positivos para el negocio, y por qué la accuracy por sí sola puede ser engañosa con clases desbalanceadas._

---
# 9. Conclusiones
**Responsable:** _todo el equipo_

### 9.1 Hallazgos principales
_..._

### 9.2 Recomendaciones de negocio
_..._

### 9.3 Limitaciones y próximos pasos
_..._